In [1]:
import argparse
import json
import os
from pathlib import Path

import numpy as np
import mediapy

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from common import (
    cumulative_episode_bounds,
    episode_shard_path,
    load_episode_frame,
    load_episode_records,
    load_json,
    load_lerobot_dataset,
    save_json_atomic,
)

from common_trace import (
    project_world_points_to_lerobot_image,
    build_bddl_language_index,
    resolve_bddl_path_for_instruction,
    compute_libero_camera_calibration
)

skill_annotations = "../../pace/openpi/data/libero-100/skill_target_traces.json"

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /home/gatech/workspace/mujoco_test/mujoco_playground/.venv/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


In [2]:
import math
def _quat2axisangle(quat):
    """
    Copied from robosuite: https://github.com/ARISE-Initiative/robosuite/blob/eafb81f54ffc104f905ee48a16bb15f059176ad3/robosuite/utils/transform_utils.py#L490C1-L512C55
    """
    # clip quaternion
    if quat[3] > 1.0:
        quat[3] = 1.0
    elif quat[3] < -1.0:
        quat[3] = -1.0

    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        # This is (close to) a zero degree rotation, immediately return
        return np.zeros(3)

    return (quat[:3] * 2.0 * math.acos(quat[3])) / den

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_depths": True}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

class LiberoEnvMaker:
    def __init__(self, suite: str,
                 render_resolution: int = 256, seed: int = 0,
                 repeats: int = 1):
        benchmark_dict = benchmark.get_benchmark_dict()
        self.task_suite = benchmark_dict[suite]()
        self.repeats = repeats
        self.render_resolution = render_resolution
        self.seed = seed

    def get_num_tasks(self):
        return self.task_suite.n_tasks

    def task_instantiations(self, task_id):
        task = self.task_suite.get_task(task_id)
        initial_states = self.task_suite.get_task_init_states(task_id)
        env, task_description = _get_libero_env(task, self.render_resolution, self.seed)
        for episode_idx in range(self.repeats):
            env.reset()
            obs = env.set_init_state(initial_states[episode_idx])
            yield obs, env, task_description


In [3]:
with open(skill_annotations) as f:
    annotation_data = json.load(f)
repo_id = annotation_data['source_repo_id']
dataset_root = os.path.dirname(skill_annotations)

records = load_episode_records(dataset_root)
episode_bounds = cumulative_episode_bounds(records)
print("loading lerobot dataset...", flush=True)
dataset = load_lerobot_dataset(repo_id, dataset_root)
print("dataset loaded", flush=True)

loading lerobot dataset...


The dataset you requested (yilin-wu/libero-100) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=yilin-wu/libero-100
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



Resolving data files:   0%|          | 0/4338 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/163 [00:00<?, ?it/s]

dataset loaded


In [4]:
# TODO: add to things that are not the first

In [5]:
dataset.meta.tasks

{0: 'put the white mug on the left plate and put the yellow and white mug on the right plate',
 1: 'turn on the stove and put the moka pot on it',
 2: 'put both the cream cheese box and the butter in the basket',
 3: 'put both moka pots on the stove',
 4: 'put the black bowl in the bottom drawer of the cabinet and close it',
 5: 'put both the alphabet soup and the cream cheese box in the basket',
 6: 'put both the alphabet soup and the tomato sauce in the basket',
 7: 'put the white mug on the plate and put the chocolate pudding to the right of the plate',
 8: 'pick up the book and place it in the back compartment of the caddy',
 9: 'put the yellow and white mug in the microwave and close it',
 10: 'pick up the white mug and place it to the right of the caddy',
 11: 'put the white mug on the plate',
 12: 'put the white bowl on the plate',
 13: 'open the top drawer of the cabinet',
 14: 'put the alphabet soup in the tray',
 15: 'put the frying pan on the cabinet shelf',
 16: 'put the ch

In [11]:
calibration_cache = {}
bddl_index = build_bddl_language_index()

def get_camera_calibration(instruction, image_size, camera_name='agentview'):
    bddl_path = resolve_bddl_path_for_instruction(
        instruction,
        bddl_index=bddl_index
    )
    cache_key = (str(bddl_path), camera_name, image_size)
    if cache_key not in calibration_cache:
        # print(f"computing EE projection camera calibration: {camera_name} from {bddl_path.name}", flush=True)
        calibration_cache[cache_key] = compute_libero_camera_calibration(
            bddl_path=bddl_path,
            camera_name=camera_name,
            image_width=image_size,
            image_height=image_size,
        )
    return calibration_cache[cache_key]

n_tasks = len(dataset.meta.tasks)
print(n_tasks)

83


In [12]:
from tqdm import tqdm
N_REPEATS = 9999
IMAGE_SIZE = 256
libero_90 = LiberoEnvMaker("libero_90", render_resolution=IMAGE_SIZE, repeats=N_REPEATS)
libero_10 = LiberoEnvMaker("libero_10", render_resolution=IMAGE_SIZE, repeats=N_REPEATS)
task_generators = [libero_90.task_instantiations(i) for i in range(libero_90.get_num_tasks())] \
                + [libero_10.task_instantiations(i) for i in range(libero_10.get_num_tasks())]

new_episode_idx = max(episode_bounds.keys())
DUMMY_TASK = "move the robot directly"
for counter, (episode_index, (start, end)) in tqdm(list(enumerate(episode_bounds.items()))):
    annot_data = annotation_data[str(episode_index)]
    obs, env, task_description = next(task_generators[counter % len(task_generators)])
    projection = get_camera_calibration(task_description, IMAGE_SIZE)
    ee_world_positions = []
    N_STEPS = min(annot_data['segments'][0]['end_step'], 60)
    init_gripper = dataset.hf_dataset[start]['actions'][-1]
    time = 0.0
    for i in range(60):
        ee_world_positions.append(obs['robot0_eef_pos'].tolist())
        row = dataset.hf_dataset[start + i]
        action = np.array(row['actions'], copy=True, dtype=np.float32)
        action[-1] = init_gripper
        obs, reward, done, info = env.step(action)
        dataset.add_frame({
            'image': np.copy(obs['agentview_image'][::-1, ::-1, :]),
            'wrist_image': np.copy(obs['robot0_eye_in_hand_image'][::-1, ::-1, :]),
            'state': np.concatenate(
                (
                    obs["robot0_eef_pos"],
                    _quat2axisangle(obs["robot0_eef_quat"]),
                    obs["robot0_gripper_qpos"]
                )
            ).astype(np.float32),
            "actions": action,
            "task": DUMMY_TASK
        })
        time += 0.1
    dataset.save_episode()
    trace_2d, depth, _ = project_world_points_to_lerobot_image(
        np.array(ee_world_positions),
        world_to_camera_transform=projection['world_to_camera_transform'],
        image_width=IMAGE_SIZE,
        image_height=IMAGE_SIZE,
        image_convention=projection.get('image_convention')
    )
    clipped_trace = np.column_stack(
        [
            np.clip(trace_2d[:, 0], 0, IMAGE_SIZE - 1),
            np.clip(trace_2d[:, 1], 0, IMAGE_SIZE - 1),
        ]
    )
    round_trace = [[int(round(x)), int(round(y))] for x, y in clipped_trace]

    fake_skill = f"MOVE_ROBOT({', '.join(f'{x:.2f}' for x in ee_world_positions[-1])})"
    entry = {
        'episode_index': new_episode_idx,
        'task_index': n_tasks,
        'instruction': DUMMY_TASK,
        'num_steps': N_STEPS,
        'fps': 10,
        'plan': f"1. {fake_skill}",
        'segments': [{
            'start_step': 0,
            'end_step': N_STEPS,
            'skill': fake_skill
        }],
        'target_traces': [{
            "skill_index": 0,
            "skill": fake_skill,
            "start_step": 0,
            "end_step": N_STEPS,
            "semantic_target": None,
            "end_effector_trace": {
                'trace': round_trace,
                'raw_trace': trace_2d.tolist(),
                'source_world_positions': ee_world_positions
            }
        }]
    }
    annotation_data[str(new_episode_idx)] = entry
    new_episode_idx += 1
# with open(skill_annotations + ".new", 'w') as f:
#     json.dump(annotation_data, f)

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


  0%|                                                                | 0/4338 [00:00<?, ?it/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

  0%|                                                     | 1/4338 [00:10<12:56:27, 10.74s/it]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

100%|█████████████████████████████████████████████████▉| 4337/4338 [10:23:59<00:09,  9.24s/it]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

100%|██████████████████████████████████████████████████| 4338/4338 [10:24:08<00:00,  8.63s/it]


In [13]:
with open(skill_annotations + ".new", 'w') as f:
    json.dump(annotation_data, f)